# Simulating IETS spectra of a spin-1/2 

In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as const

from spinfinity import dIdV, SpinSys, bath, hamil


%load_ext autoreload
%autoreload 2

## Setting up the Hamiltonian

For a magnetic field along $z$, Spinfinity uses

$$
H=-g\mu_B B S_z.
$$

The two eigenstates have $m_z=\pm 1/2$, with an energy separation

$$
\Delta E=g\mu_B B.
$$

At $B=5$ T and $g=2$, this gives $\Delta E\approx0.579$ meV.
The corresponding tunneling excitation thresholds occur near
$V=\pm0.579$ mV.

Hamiltonian energies are expressed in meV, magnetic fields in T,
temperature in K, and conductances in S.

In [ ]:
spin = SpinSys([0.5])

# Spin Hamiltonian parameters
spin.gfactorvector = [2.0]
spin.B = [0.0, 0.0, 5.0]  # T
spin.BTip = [0.0, 0.0, 0.0]

# Temperature and scattering parameters
spin.T = 1.0  # K
spin.G_st = 1e-9  # S
spin.G_ss = [1e-7]  # S
spin.G_tt = 0
spin.U = np.array([0.0])
spin.b0 = 0.0
spin.TipPolarization = [0.0, 0.0, 0.0]
spin.SamplePolarization = [0.0, 0.0, 0.0]

hamil.calc_EigEnergies(spin, rebuildH=True)
hamil.calc_EigStates(spin)

gap = spin.E_All[1] - spin.E_All[0]

print(f"Relative eigenenergies: {spin.E_All} meV")
print(f"Excitation thresholds: ±{gap:.4f} mV")

hamil.plot_EigenMatrix(spin);
hamil.plot_ZeemanDiagramm(spin);

# Common voltage grid and plotting colors
V_array = np.linspace(-2.0, 2.0, 1201)  # mV
colors = ("#636EFA", "#EF553B", "#00CC96")

## Basic second-order IETS

Tunneling electrons can excite the spin once their available energy
exceeds the excitation energy:

$$
e|V|\geq\Delta E.
$$

Opening this additional transport channel produces steps in $dI/dV$.
The dashed lines mark the excitation thresholds; finite temperature
broadens the steps.

`calc_IETS_2nd` assumes thermal spin populations. It does not include
population changes caused by the tunneling current.

We use `norm=False` and display the conductance in nS.

In [ ]:
return_dict = dIdV.plot_IETS(
    spin, 
    V_array=V_array, 
    norm=False, 
    order="2nd"  # defines the order of perturbation used
)

## Varying individual parameters

Only the indicated parameter is changed.

### Temperature

Increasing temperature broadens the conductance steps and increases
the equilibrium occupation of the excited state.

The thermal population ratio is

$$
\frac{p_{\mathrm{exc}}}{p_{\mathrm{ground}}}
=
\exp\left(-\frac{\Delta E}{k_B T}\right).
$$

Temperature changes the populations and spectral shape, but not the
Hamiltonian's excitation energy.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)

for temperature in (0.3, 1.0, 3.0):
    spin.T = temperature

    V, conductance = dIdV.calc_IETS_2nd(
        spin, V_array=V_array, norm=False
    )
    ax.plot(
        V, conductance * 1e9, label=f"T = {temperature:g} K"
    )

# Restore temperature and thermal populations.
spin.T = 1.0
bath.calc_Populations(spin, AllowPumping=False)

ax.set(
    title="Temperature: broadening and thermal occupation",
    xlabel="Bias voltage (mV)",
    ylabel="dI/dV (nS)",
)
ax.legend()
plt.show()

### Tip–sample coupling

`G_st` sets the conductance scale for electron transport between
tip and sample. With the used scaling `G_st` corresponds to the zero-voltage conductance without inelastic excitations.

In [ ]:
spin.G_ss = [1e-9]
spin.G_tt = 0

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)

for index, conductance_nS in enumerate((1.0, 2.0, 4.0)):
    spin.G_st = conductance_nS * 1e-9

    V, conductance = dIdV.calc_IETS_2nd(
        spin,
        V_array=V_array,
        norm=False,
    )
    label = f"Gst = {conductance_nS:g} nS"

    # Explicit division removes the overall conductance scale.
    ax.plot(
        V,
        conductance,
        label=label,
    )
    

spin.G_st = 1e-9
spin.G_ss = [1e-7]

ax.set(
    title="Tip-sample coupling at fixed substrate coupling",
    ylabel="dI/dV (S)",
)
ax.legend()
plt.show()

### Magnetic field

The Zeeman splitting increases linearly with magnetic field:

$$
\Delta E=g\mu_B B.
$$

Consequently, increasing the field moves the excitation thresholds
to larger absolute voltages. It also changes the thermal population
ratio at fixed temperature.

After changing `B`, we explicitly rebuild and diagonalize the
Hamiltonian using `rebuildH=True`.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)

for field in (2.5, 5.0, 7.5):
    spin.B = [0.0, 0.0, field]
    hamil.calc_EigEnergies(spin, rebuildH=True)

    V, conductance = dIdV.calc_IETS_2nd(
        spin, V_array=V_array, norm=False
    )
    ax.plot(V, conductance * 1e9, label=f"B = {field:g} T")

# Restore the field, eigenenergies, and thermal populations.
spin.B = [0.0, 0.0, 5.0]
hamil.calc_EigEnergies(spin, rebuildH=True)
bath.calc_Populations(spin, AllowPumping=False)

ax.set(
    title="Magnetic field: shifting the excitation thresholds",
    xlabel="Bias voltage (mV)",
    ylabel="dI/dV (nS)",
)
ax.legend()
plt.show()

### Potential scattering: U

`U` is a dimensionless potential-scattering amplitude. It contributes
to elastic scattering without changing the localized spin state.

For this unpolarized setup, increasing `U` reduces the relative
inelastic step contrast without shifting the excitation thresholds.

Spinfinity normalizes the scattering contributions using the
ground-state elastic matrix element. Therefore, at fixed `G_st`,
changing `U` should not be interpreted as simply adding an independent
absolute conductance.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)

for potential_scattering in (0.0, 0.25, 0.5):
    spin.U = np.array([potential_scattering])

    V, conductance = dIdV.calc_IETS_2nd(
        spin, V_array=V_array, norm=False
    )
    ax.plot(
        V,
        conductance * 1e9,
        label=f"U = {potential_scattering:g}",
    )

spin.U = np.array([0.0])

ax.set(
    title="Potential scattering: changing the inelastic contrast",
    xlabel="Bias voltage (mV)",
    ylabel="dI/dV (nS)",
)
ax.legend()
plt.show()

### Spin-independent background: b0

`b0` assigns a fraction of the conductance to a separate
spin-independent background:

$$
G_{\mathrm{background}}=b_0G_{st}.
$$

The remaining spin-related contribution is scaled by $1-b_0$.
Increasing `b0` therefore reduces the conductance steps without
changing their excitation energies.

Unlike `U`, this background is added separately rather than included
in the microscopic scattering amplitude.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)

for background in (0.0, 0.5, 0.9):
    spin.b0 = background

    V, conductance = dIdV.calc_IETS_2nd(
        spin, V_array=V_array, norm=False
    )
    ax.plot(
        V, conductance * 1e9, label=f"b0 = {background:g}"
    )

spin.b0 = 0.0

ax.set(
    title="Spin-independent background: reducing the step contrast",
    xlabel="Bias voltage (mV)",
    ylabel="dI/dV (nS)",
)
ax.legend()
plt.show()

### Tip polarization

The tip polarization describes the imbalance between its electron
spin populations:

$$
\eta_z =
\frac{n_\uparrow-n_\downarrow}{n_\uparrow+n_\downarrow}.
$$

We compare an unpolarized tip with tips polarized along positive
and negative $z$, while keeping the sample unpolarized.

Spin-selective scattering produces different conductance step heights
at positive and negative bias. For this setup, reversing the tip
polarization mirrors the spectrum about zero bias.

Changing `TipPolarization` does not change the Hamiltonian or its
excitation energies. It is distinct from changing `BTip`.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)

for polarization in (-0.3, 0.0, 0.8):
    spin.TipPolarization = [0.0, 0.0, polarization]

    V, conductance = dIdV.calc_IETS_2nd(
        spin, V_array=V_array, norm=False
    )
    ax.plot(
        V,
        conductance * 1e9,
        label=rf"$\eta_z = {polarization:+.1f}$",
    )

# Restore the unpolarized baseline for the following sections.
spin.TipPolarization = [0.0, 0.0, 0.0]

ax.set(
    title="Tip polarization: bias-dependent step contrast",
    xlabel="Bias voltage (mV)",
    ylabel="dI/dV (nS)",
)
ax.legend()
plt.show()

## Second-order IETS with rate equations

`calc_IETS_2ndR` calculates voltage-dependent steady-state populations.
Tunneling can therefore drive the spin away from thermal equilibrium.

The upper panels show differential conductance. The lower panels show
the excited-state population, illustrating how population changes
affect the spectrum.

### Substrate coupling

`G_ss` controls scattering between the spin and substrate electrons.
These processes exchange energy with the spin but do not transfer
charge between tip and sample.

At fixed `G_st`, stronger substrate coupling keeps the populations
closer to thermal equilibrium. Weaker substrate coupling allows
larger tunneling-induced deviations.

We vary $G_{ss}/G_{st}$ and disable tip–tip scattering to isolate
this competition. The dashed curve is the thermal second-order
reference.

In [ ]:
spin.G_st = 1e-9
spin.G_tt = 0

fig, axes = plt.subplots(
    2, 1, sharex=True, figsize=(7, 6), constrained_layout=True
)

V, thermal = dIdV.calc_IETS_2nd(
    spin, V_array=V_array, norm=False
)
axes[0].plot(
    V, thermal * 1e9, "k--", label="Thermal reference"
)

for index, ratio in enumerate((0.01, 1.0, 100.0)):
    spin.G_ss = [ratio * spin.G_st]

    V, conductance, populations = dIdV.calc_IETS_2ndR(
        spin,
        V_array=V_array,
        AllowPumping=True,
        norm=False,
    )
    label = f"Gss/Gst = {ratio:g}"

    axes[0].plot(
        V, conductance * 1e9, color=colors[index], label=label
    )
    axes[1].plot(
        V, populations[1], color=colors[index]
    )

spin.G_ss = [1e-7]

axes[0].set(
    title="Substrate coupling: pumping versus relaxation",
    ylabel="dI/dV (nS)",
)
axes[1].set(
    xlabel="Bias voltage (mV)",
    ylabel="Excited-state population",
)
axes[0].legend()
plt.show()

### Tip–tip scattering

Tip–tip scattering exchanges energy between the spin and electrons
within the tip reservoir. It does not directly transfer charge
between tip and sample.

In the current implementation, `G_tt` is a switch:

- `G_tt = 0`: tip–tip scattering disabled.
- `G_tt = 1`: tip–tip scattering enabled.

Its effective conductance scale is determined by

$$
G_{tt,\mathrm{eff}}
=
\frac{\left[(1-b_0)G_{st}\right]^2}{G_{ss}}.
$$

For the unpolarized tip used here, enabling this channel provides an
additional route toward thermal equilibrium and reduces the
tunneling-induced excited-state population.

Because its strength depends on `G_ss`, varying substrate coupling
with tip–tip scattering enabled would change two scattering
contributions simultaneously.

In [ ]:
spin.G_st = 1e-9
spin.G_ss = [0.1 * spin.G_st]

fig, axes = plt.subplots(
    2, 1, sharex=True, figsize=(7, 6), constrained_layout=True
)

for index, enabled in enumerate((0, 1)):
    spin.G_tt = enabled

    V, conductance, populations = dIdV.calc_IETS_2ndR(
        spin,
        V_array=V_array,
        AllowPumping=True,
        norm=False,
    )
    label = (
        "Tip-tip scattering on"
        if enabled
        else "Tip-tip scattering off"
    )

    axes[0].plot(
        V, conductance * 1e9, color=colors[index], label=label
    )
    axes[1].plot(
        V, populations[1], color=colors[index]
    )

spin.G_ss = [1e-7]
spin.G_tt = 0

axes[0].set(
    title="Tip-tip scattering: an additional relaxation channel",
    ylabel="dI/dV (nS)",
)
axes[1].set(
    xlabel="Bias voltage (mV)",
    ylabel="Excited-state population",
)
axes[0].legend()
plt.show()